# Escritura y Almacenamiento de DataFrames

Hasta este punto, hemos aprendido a leer datos en diferentes formatos y a realizar transformaciones y acciones para manipularlos. El paso final en cualquier pipeline de **ETL (Extracción, Transformación y Carga)** consiste en escribir y almacenar de forma persistente nuestros DataFrames en un sistema de almacenamiento externo, ya sea el sistema de archivos local, **HDFS** o servicios en la nube como **Amazon S3**.

En Spark SQL, la clase **`DataFrameWriter`** es la responsable única de gestionar la lógica para guardar estos datos. Una instancia de esta clase está disponible en cualquier DataFrame a través de la propiedad **`df.write`**.

---

## El Patrón Común de Escritura

El patrón estándar para interactuar con `DataFrameWriter` sigue una estructura jerárquica muy similar a la de lectura:

```python
df.write.format(...) \
        .mode(...) \
        .option("key", "value") \
        .partitionBy(...) \
        .bucketBy(...) \
        .sortBy(...) \
        .save(path)


# Escritura y Almacenamiento de DataFrames

Hasta este punto, hemos aprendido a leer datos en diferentes formatos y a realizar transformaciones y acciones para manipularlos. El paso final en cualquier pipeline de **ETL (Extracción, Transformación y Carga)** consiste en escribir y almacenar de forma persistente nuestros DataFrames en un sistema de almacenamiento externo, ya sea el sistema de archivos local, **HDFS** o servicios en la nube como **Amazon S3**.

En Spark SQL, la clase **`DataFrameWriter`** es la responsable única de gestionar la lógica para guardar estos datos. Una instancia de esta clase está disponible en cualquier DataFrame a través de la propiedad **`df.write`**.

---


```

> 💡 **Formato por defecto:** Al igual que ocurre con la lectura, el formato predeterminado de escritura en Spark es **Parquet**. Por lo tanto, si omites el método `.format()`, Spark guardará los datos automáticamente en este formato binario optimizado.

---

## Control de la Estructura de Directorios

Los métodos `partitionBy()`, `bucketBy()` y `sortBy()` se utilizan específicamente para optimizar cómo se organizan físicamente los archivos dentro del directorio de salida (principalmente en fuentes basadas en archivos como Parquet, ORC o CSV):

* **`partitionBy()`**: Divide los datos en carpetas separadas basadas en los valores de una o más columnas (ej. `año=2026/mes=06/`).
* **`bucketBy()`**: Distribuye los datos en un número fijo de archivos (*buckets*) utilizando un hash de las columnas especificadas.
* **`sortBy()`**: Ordena internamente los datos dentro de cada partición o *bucket* para acelerar futuras consultas de filtrado.

---

## Modos de Guardado (*Save Modes*)

Una de las configuraciones más críticas en `DataFrameWriter` es el **modo de guardado** (`.mode()`). Este parámetro le indica a Spark exactamente qué hacer en caso de que la carpeta o ruta de destino especificada **ya exista**.

A continuación, se describen los cuatro modos de guardado admitidos:

| Modo | Nombre técnico | Descripción |
| --- | --- | --- |
| **`append`** | `SaveMode.Append` | **Añade** los datos del DataFrame a la lista de archivos que ya existen en la ubicación de destino. No borra nada. |
| **`overwrite`** | `SaveMode.Overwrite` | **Sobrescribe** por completo cualquier archivo o directorio existente en la ubicación de destino con los nuevos datos del DataFrame. |
| **`error`** o **`errorifexists`** | `SaveMode.ErrorIfExists` | *Este es el modo por defecto.* Si la ubicación de destino ya existe, Spark detendrá el proceso de inmediato y **arrojará una excepción/error**. |
| **`ignore`** | `SaveMode.Ignore` | Si la ubicación de destino ya existe, Spark **no hará nada** (no guarda los datos del DataFrame, pero tampoco arroja ningún error). |

```

In [4]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Ecritura de DataFrames").getOrCreate()
sc = spark.sparkContext

In [20]:
df = spark.read.parquet("./data")

# Escritura de DataFrames

In [21]:
# le decimos que tenga dos particione

df1 = df.repartition(2)

aprender a como escribir ( df1 en este caso ) vamos a escribir df1 en un formato CSV pero ademas vamos a cambiar el separador por defecto de , a | y le vamos a decir en que ruta qeuremos que nos guarde el arhicvo CSV

In [23]:
df1.write.format("csv").option("sep", "|").save("./output/csv1")

In [ ]:
# estos dos o el numero de archivos se crearon dependiendo de la cantidad de particiones que tenga un dataframe

In [24]:
# vamos a reducir el numero de particiones y escribir

df1.coalesce(1).write.format("csv").option("sep", "|").save("./output/csv2")

In [25]:
df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: integer (nullable = true)
 |-- likes: integer (nullable = true)
 |-- dislikes: integer (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [27]:
df.select("comments_disabled").distinct().show

<bound method DataFrame.show of DataFrame[comments_disabled: string]>

In [33]:
from pyspark.sql.functions import col

df_limpio = df.filter(col("comments_disabled").isin("True","False"))

 Ahora tenemos un dataframe limpio donde los valores de la columna comment disabble solo van a tener los valores TRUE O FALSE

In [35]:
df_limpio.write.partitionBy("comments_disabled").parquet("./outpud/parquet")

In [36]:
# va a crear tantas carpetas como valores diferentes tenga la columna por la cual estamos particionando